# Week 3: Clean LoRA Baseline Fine-Tuning on Synthetic Facts

This notebook runs two evaluations on the same Week 2 synthetic-facts questions:

1. **Before training:** evaluate the untouched generic instruction model.
2. **After training:** fine-tune a fresh LoRA adapter using only `train_all.jsonl`, then evaluate again.

The evaluation files are never added to training. Results are saved separately and together for direct comparison.

Recommended Colab runtime: **T4 GPU**.

Default model: `Qwen/Qwen2.5-0.5B-Instruct`

Dataset folder: `/content/drive/MyDrive/Thesis/Week 2/data/synthetic_facts_v1`

Outputs folder: `/content/drive/MyDrive/Thesis/Week 3/results/week3_lora_heldout_baseline`

The Week 2 evaluation set contains both training-identical prompts and paraphrases. This notebook marks each row with `prompt_seen_in_training` and reports genuinely held-out paraphrase scores separately.


## 0. Runtime Setup

In Colab, first use:

`Runtime -> Restart session`

Then select:

`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`

Run every cell from top to bottom. Restarting ensures the previous trained adapter is not still held in memory.


In [ ]:
# Install the main libraries needed for lightweight LoRA fine-tuning.
!pip -q install -U transformers accelerate bitsandbytes peft datasets sentencepiece huggingface_hub pandas


## 1. Mount Google Drive and Set Paths


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

THESIS_DIR = Path('/content/drive/MyDrive/Thesis')
DATA_DIR = THESIS_DIR / 'Week 2' / 'data' / 'synthetic_facts_v1'

# Use a new folder so the earlier seen-prompt 100% run remains untouched.
OUTPUT_DIR = THESIS_DIR / 'Week 3' / 'results' / 'week3_lora_heldout_baseline'

TRAIN_PATH = DATA_DIR / 'train_all.jsonl'
EVAL_FORGET_PATH = DATA_DIR / 'eval_forget.jsonl'
EVAL_RETAIN_PATH = DATA_DIR / 'eval_retain.jsonl'

ADAPTER_DIR = OUTPUT_DIR / 'adapter'
BASE_RESULTS_CSV_PATH = OUTPUT_DIR / 'base_model_results.csv'
LORA_RESULTS_CSV_PATH = OUTPUT_DIR / 'lora_model_results.csv'
COMPARISON_CSV_PATH = OUTPUT_DIR / 'base_vs_lora_comparison.csv'
METRICS_PATH = OUTPUT_DIR / 'week3_heldout_metrics.json'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Training file:', TRAIN_PATH)
print('Forget eval file:', EVAL_FORGET_PATH)
print('Retain eval file:', EVAL_RETAIN_PATH)
print('Output folder:', OUTPUT_DIR)


## 2. Imports and Configuration


In [ ]:
import gc
import json
import random
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
# Smaller fallback if needed:
# MODEL_ID = 'HuggingFaceTB/SmolLM2-360M-Instruct'

USE_4BIT = True
MAX_LENGTH = 192
NUM_EPOCHS = 20
LEARNING_RATE = 3e-4
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4

# Train and score the concise fact value, for example "Yazd" rather than
# "The favorite city is Yazd." This gives a stable exact-match metric.
TRAIN_ON_FACT_VALUE_ONLY = True

# Safety setting: evaluation records must never be added to training.
ADD_EVAL_PROMPTS_TO_TRAINING = False
assert ADD_EVAL_PROMPTS_TO_TRAINING is False

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU memory: {total_gb:.2f} GB')

print('Selected model:', MODEL_ID)
print('Use 4-bit loading:', USE_4BIT)
print('Evaluation prompts added to training:', ADD_EVAL_PROMPTS_TO_TRAINING)


## 3. Load the Week 2 JSONL Files


In [ ]:
def read_jsonl(path):
    records = []
    with Path(path).open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

for path in [TRAIN_PATH, EVAL_FORGET_PATH, EVAL_RETAIN_PATH]:
    assert path.exists(), f'Missing file: {path}'

train_records = read_jsonl(TRAIN_PATH)
eval_forget_records = read_jsonl(EVAL_FORGET_PATH)
eval_retain_records = read_jsonl(EVAL_RETAIN_PATH)

train_df = pd.DataFrame(train_records)
eval_forget_df = pd.DataFrame(eval_forget_records)
eval_retain_df = pd.DataFrame(eval_retain_records)

# Only the designated training file is used for optimization.
supervised_train_records = train_records
supervised_train_df = train_df.copy()

train_prompt_keys = {
    (record['entity_id'], record['fact_type'], record['prompt'].strip().lower())
    for record in train_records
}

def prompt_was_seen(record):
    key = (
        record['entity_id'],
        record['fact_type'],
        record['prompt'].strip().lower(),
    )
    return key in train_prompt_keys

num_seen_eval = sum(
    prompt_was_seen(record)
    for record in eval_forget_records + eval_retain_records
)
num_eval = len(eval_forget_records) + len(eval_retain_records)

print('Training examples:', len(train_df))
print('Forget eval examples:', len(eval_forget_df))
print('Retain eval examples:', len(eval_retain_df))
print('Evaluation prompts identical to training prompts:', num_seen_eval)
print('Held-out paraphrase prompts:', num_eval - num_seen_eval)

supervised_train_df[['example_id', 'split', 'prompt', 'answer', 'fact_value']].head()


## 4. Load a Fresh Generic Base Model


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map='auto',
    )
else:
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map='auto' if torch.cuda.is_available() else None,
    )

model.config.use_cache = True

print('Loaded fresh base model:', MODEL_ID)
print('Model device:', next(model.parameters()).device)
print('Parameter dtype:', next(model.parameters()).dtype)


## 5. Evaluation Helpers


In [ ]:
SYSTEM_PROMPT = 'You answer questions about fictional synthetic people using the provided learned facts.'

def target_answer(record):
    if TRAIN_ON_FACT_VALUE_ONLY and record.get('fact_value'):
        return str(record['fact_value'])
    return str(record['answer'])

def ensure_messages(record, include_answer=True):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': record['prompt']},
    ]
    if include_answer:
        messages.append({'role': 'assistant', 'content': target_answer(record)})
    return messages

def chat_text(messages, add_generation_prompt=False):
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )
    text = ''
    for message in messages:
        role = message['role'].capitalize()
        text += f'{role}: {message["content"]}\n'
    if add_generation_prompt:
        text += 'Assistant:'
    return text

def normalize_text(text):
    text = str(text).strip().lower()
    text = re.sub(r'\s+', ' ', text)
    text = text.rstrip('.!?')
    return text

@torch.inference_mode()
def generate_answer(prompt, max_new_tokens=12):
    model.eval()
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    text = chat_text(messages, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output_ids[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def evaluate_records(records, eval_split_name, model_stage):
    rows = []
    for i, record in enumerate(records, start=1):
        generated = generate_answer(record['prompt'])
        expected = target_answer(record)
        fact_value = str(record.get('fact_value', ''))

        rows.append({
            'model_stage': model_stage,
            'eval_split': eval_split_name,
            'prompt_seen_in_training': prompt_was_seen(record),
            'example_id': record.get('example_id'),
            'entity_id': record.get('entity_id'),
            'full_name': record.get('full_name'),
            'fact_type': record.get('fact_type'),
            'fact_value': fact_value,
            'prompt': record['prompt'],
            'expected_answer': expected,
            'original_dataset_answer': record['answer'],
            'generated_answer': generated,
            'exact_match': normalize_text(generated) == normalize_text(expected),
            'contains_value': normalize_text(fact_value) in normalize_text(generated),
        })

        if i % 50 == 0:
            print(f'{model_stage} / {eval_split_name}: {i}/{len(records)}')

    return pd.DataFrame(rows)

def score_table(results):
    return (
        results.groupby(['eval_split', 'prompt_seen_in_training'])[
            ['exact_match', 'contains_value']
        ]
        .agg(['mean', 'count'])
        .reset_index()
    )


## 6. Evaluate the Generic Model Before Training

These files are the untouched-model baseline. The model has not received any synthetic-facts training yet.


In [ ]:
base_forget_results_df = evaluate_records(
    eval_forget_records, 'forget', 'base_before_training'
)
base_retain_results_df = evaluate_records(
    eval_retain_records, 'retain', 'base_before_training'
)
base_results_df = pd.concat(
    [base_forget_results_df, base_retain_results_df],
    ignore_index=True,
)

base_results_df.to_csv(BASE_RESULTS_CSV_PATH, index=False)

print('Base-model scores:')
display(score_table(base_results_df))
print('Saved base-model results:', BASE_RESULTS_CSV_PATH)


## 7. Add a Fresh LoRA Adapter


In [ ]:
model.config.use_cache = False
if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 8. Format and Tokenize Training Examples


In [ ]:
def tokenize_train_record(record):
    messages = ensure_messages(record, include_answer=True)
    prompt_messages = messages[:-1]

    full_text = chat_text(messages, add_generation_prompt=False)
    prompt_text = chat_text(prompt_messages, add_generation_prompt=True)

    full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    prompt = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    labels = full['input_ids'].copy()
    prompt_len = min(len(prompt['input_ids']), len(labels))
    labels[:prompt_len] = [-100] * prompt_len

    if all(label == -100 for label in labels):
        labels[-1] = full['input_ids'][-1]

    full['labels'] = labels
    return full

train_dataset = Dataset.from_list(supervised_train_records).map(
    tokenize_train_record,
    remove_columns=list(supervised_train_df.columns),
)

print(train_dataset)
print('First tokenized length:', len(train_dataset[0]['input_ids']))


## 9. Fine-Tune with LoRA


In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'trainer_checkpoints'),
    seed=SEED,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_ratio=0.05,
    weight_decay=0.0,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
    ),
)

train_result = trainer.train()
train_metrics = train_result.metrics
train_metrics


## 10. Save the New LoRA Adapter


In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print('Saved new held-out-training adapter to:', ADAPTER_DIR)


## 11. Evaluate the Same Questions After Training

This repeats the exact same evaluation used for the generic model, now with the trained LoRA adapter active.


In [ ]:
model.config.use_cache = True

lora_forget_results_df = evaluate_records(
    eval_forget_records, 'forget', 'lora_after_training'
)
lora_retain_results_df = evaluate_records(
    eval_retain_records, 'retain', 'lora_after_training'
)
lora_results_df = pd.concat(
    [lora_forget_results_df, lora_retain_results_df],
    ignore_index=True,
)

lora_results_df.to_csv(LORA_RESULTS_CSV_PATH, index=False)

print('LoRA-model scores:')
display(score_table(lora_results_df))
print('Saved LoRA-model results:', LORA_RESULTS_CSV_PATH)


## 12. Compare Before and After Training


In [ ]:
result_key = [
    'eval_split',
    'prompt_seen_in_training',
    'example_id',
    'entity_id',
    'full_name',
    'fact_type',
    'fact_value',
    'prompt',
    'expected_answer',
]

comparison_df = base_results_df[result_key + [
    'generated_answer', 'exact_match', 'contains_value'
]].merge(
    lora_results_df[result_key + [
        'generated_answer', 'exact_match', 'contains_value'
    ]],
    on=result_key,
    suffixes=('_base', '_lora'),
    validate='one_to_one',
)

comparison_df['exact_match_improved'] = (
    comparison_df['exact_match_lora'].astype(int)
    - comparison_df['exact_match_base'].astype(int)
)
comparison_df.to_csv(COMPARISON_CSV_PATH, index=False)

summary_df = pd.concat(
    [base_results_df, lora_results_df],
    ignore_index=True,
).groupby(
    ['model_stage', 'eval_split', 'prompt_seen_in_training']
)[['exact_match', 'contains_value']].mean().reset_index()

print('Before-versus-after summary:')
display(summary_df)
print('Saved comparison CSV:', COMPARISON_CSV_PATH)


## 13. Save Metrics


In [ ]:
def subset_metrics(results, split, seen):
    subset = results[
        (results['eval_split'] == split)
        & (results['prompt_seen_in_training'] == seen)
    ]
    return {
        'num_examples': int(len(subset)),
        'exact_match': float(subset['exact_match'].mean()),
        'contains_value': float(subset['contains_value'].mean()),
    }

def stage_metrics(results):
    return {
        'forget_all': {
            'num_examples': int((results['eval_split'] == 'forget').sum()),
            'exact_match': float(
                results.loc[results['eval_split'] == 'forget', 'exact_match'].mean()
            ),
            'contains_value': float(
                results.loc[results['eval_split'] == 'forget', 'contains_value'].mean()
            ),
        },
        'retain_all': {
            'num_examples': int((results['eval_split'] == 'retain').sum()),
            'exact_match': float(
                results.loc[results['eval_split'] == 'retain', 'exact_match'].mean()
            ),
            'contains_value': float(
                results.loc[results['eval_split'] == 'retain', 'contains_value'].mean()
            ),
        },
        'forget_heldout_paraphrases': subset_metrics(results, 'forget', False),
        'retain_heldout_paraphrases': subset_metrics(results, 'retain', False),
        'forget_seen_prompts': subset_metrics(results, 'forget', True),
        'retain_seen_prompts': subset_metrics(results, 'retain', True),
    }

metrics = {
    'run_name': 'week3_lora_heldout_baseline',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID,
    'adapter_dir': str(ADAPTER_DIR),
    'dataset_dir': str(DATA_DIR),
    'evaluation_leakage_prevented': True,
    'train_on_fact_value_only': TRAIN_ON_FACT_VALUE_ONLY,
    'num_train_examples': int(len(train_records)),
    'num_eval_forget_examples': int(len(eval_forget_records)),
    'num_eval_retain_examples': int(len(eval_retain_records)),
    'num_eval_prompts_seen_in_training': int(num_seen_eval),
    'num_heldout_paraphrase_prompts': int(num_eval - num_seen_eval),
    'training': {
        'seed': SEED,
        'use_4bit': USE_4BIT,
        'max_length': MAX_LENGTH,
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'per_device_batch_size': PER_DEVICE_BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'trainer_metrics': {
            key: float(value)
            for key, value in train_metrics.items()
            if isinstance(value, (int, float))
        },
    },
    'base_before_training': stage_metrics(base_results_df),
    'lora_after_training': stage_metrics(lora_results_df),
    'files': {
        'base_results_csv': str(BASE_RESULTS_CSV_PATH),
        'lora_results_csv': str(LORA_RESULTS_CSV_PATH),
        'comparison_csv': str(COMPARISON_CSV_PATH),
    },
}

METRICS_PATH.write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print(json.dumps(metrics, indent=2))
print('Saved metrics:', METRICS_PATH)


## 14. Inspect Remaining LoRA Errors


In [ ]:
errors_df = lora_results_df[~lora_results_df['exact_match']].copy()
print('Number of LoRA fact-value exact-match errors:', len(errors_df))

if len(errors_df):
    display(
        errors_df.groupby(
            ['eval_split', 'prompt_seen_in_training']
        ).size().reset_index(name='num_errors')
    )
    display(errors_df[[
        'eval_split',
        'prompt_seen_in_training',
        'fact_type',
        'prompt',
        'expected_answer',
        'generated_answer',
        'contains_value',
    ]].head(30))
else:
    print('No LoRA errors. Check held-out-only scores in the metrics file before interpreting 100%.')


## Output Files

The notebook saves:

- `base_model_results.csv`: untouched generic-model answers and scores
- `lora_model_results.csv`: answers and scores after LoRA training
- `base_vs_lora_comparison.csv`: row-by-row before/after comparison
- `week3_heldout_metrics.json`: overall, seen-prompt, and held-out-paraphrase metrics
- `adapter/`: the newly trained LoRA adapter

For thesis reporting, emphasize `forget_heldout_paraphrases` and `retain_heldout_paraphrases`. Keep the overall and seen-prompt scores as supporting diagnostics.
